# Library 

Import necessary libraries for audio processing and machine learning

In [ ]:
import os
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import librosa
import librosa.display

import soundfile as sf
import warnings
warnings.filterwarnings('ignore')

# Load Data

In [ ]:
base_dir = "dataset"
data = os.path.join(base_dir, "part-rusak")
classes = ["Clutch-Shoe", "Conecting-Rod", "Drive-Belt", "Piston", "Tensioner", "Slider", "Roller"]

for cls in classes:
    path = os.path.join(data, cls)
    
    if os.path.exists(path):
        jml_file = len(os.listdir(path))
        print(f"{cls}: {jml_file}")
    else:
        print(f"{cls}: Folder tidak ditemukan")

# EDA

## Informasi Distribusi Durasi Data

In [ ]:
durasi_dict = {}

print(f"{'KELAS':<15} | {'FILE':<5} | {'RATA-RATA':<10} | {'MIN':<6} | {'MAX':<6}")
print("-" * 55)

for cls in classes:
    path = os.path.join(data, cls)
    if not os.path.exists(path):
        continue
        
    files = [f for f in os.listdir(path) if f.endswith(('.wav', '.mp3'))]
    durasi_list = [librosa.get_duration(path=os.path.join(path, f)) for f in files]
    
    if durasi_list:
        durasi_dict[cls] = durasi_list
        print(f"{cls:<15} | {len(files):<5} | {np.mean(durasi_list):<8.2f}s | {np.min(durasi_list):<4.2f}s | {np.max(durasi_list):<4.2f}s")

## Distribusi Kelas

In [ ]:
durasi_dict = {}
jumlah_file_dict = {}


for cls in classes:
    path = os.path.join(data, cls)
    if not os.path.exists(path):
        continue
        
    files = [f for f in os.listdir(path) if f.endswith(('.wav', '.mp3'))]
    durasi_list = [librosa.get_duration(path=os.path.join(path, f)) for f in files]
    
    if durasi_list:
        durasi_dict[cls] = durasi_list
        jumlah_file_dict[cls] = len(files)

plt.figure(figsize=(10, 5))
plt.bar(jumlah_file_dict.keys(), jumlah_file_dict.values(), color='skyblue', edgecolor='black')
plt.title("Jumlah File per Kelas Komponen")
plt.ylabel("Jumlah File")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.boxplot(durasi_dict.values(), tick_labels=durasi_dict.keys(), patch_artist=True, boxprops=dict(facecolor='lightblue'))
plt.title("Sebaran Durasi Audio Tiap Komponen Mesin")
plt.ylabel("Durasi (Detik)")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## Detail Kelas

In [ ]:
for cls in classes:
    path = os.path.join(data, cls)
    if not os.path.exists(path):
        continue
        
    print(f"\n--- Kelas: {cls} ---")
    
    files = sorted([f for f in os.listdir(path) if f.endswith(('.wav', '.mp3', '.m4a'))])
    
    for f in files:
        file_path = os.path.join(path, f)
        durasi = librosa.get_duration(path=file_path)
        
        nama_file = os.path.splitext(f)[0]
        
        if durasi >= 60:
            menit = int(durasi // 60)
            detik = int(durasi % 60)
            print(f"{nama_file} {menit} menit {detik} detik")
        else:
            print(f"{nama_file} {durasi:.2f} detik")

# Data Preprocessing

### Sorting 

In [ ]:
base_dir = "Dataset"
data = os.path.join(base_dir, "part-rusak")
classes = ["Clutch-Shoe", "Conecting-Rod", "Drive-Belt", "Piston", "Tensioner", "Slider", "Roller"]
out_dir = os.path.join(base_dir, "rename")
os.makedirs(out_dir, exist_ok=True)

print("rename file...\n")
for cls in classes:
    in_path = os.path.join(data, cls)
    out_path = os.path.join(out_dir, cls)
    
    if not os.path.exists(in_path):
        continue
        
    os.makedirs(out_path, exist_ok=True)
    
    files = sorted([f for f in os.listdir(in_path) if f.endswith(('.wav', '.mp3', '.m4a'))])
    
    for i, f in enumerate(files, start=1):
        ext = os.path.splitext(f)[1]
        
        nama_baru = f"{cls.lower()}-{i:03d}{ext}"
        
        path_lama = os.path.join(in_path, f)
        path_baru = os.path.join(out_path, nama_baru)
        
        shutil.copy(path_lama, path_baru)
        
print(f"Selesai!")

### Menyeragamkan durasi

In [ ]:

base_dir = "dataset"
data = os.path.join(base_dir, "rename")
out_dir = os.path.join(base_dir, "durasi-2s")
classes = ["Clutch-Shoe", "Conecting-Rod", "Drive-Belt", "Piston", "Tensioner", "Slider", "Roller"]

os.makedirs(out_dir, exist_ok=True)
target_len = 16000 * 2  

print("Memulai proses preprocessing (Standarisasi 3 Detik)...\n")
print(f"{'KELAS':<15} | {'SEBELUM':<10} | {'SESUDAH':<10}")
print("-" * 43)

for cls in classes:
    in_path = os.path.join(data, cls)
    out_path = os.path.join(out_dir, cls)
    
    if not os.path.exists(in_path): 
        continue
        
    os.makedirs(out_path, exist_ok=True)
    files = [f for f in os.listdir(in_path) if f.endswith(('.wav', '.mp3', '.m4a'))]
    
    jml_sebelum = len(files)
    
    for f in files:
        audio, sr = librosa.load(os.path.join(in_path, f), sr=16000)
        nama_asli = os.path.splitext(f)[0]
        
        if len(audio) < target_len:
            audio_pad = np.pad(audio, (0, target_len - len(audio)))
            sf.write(os.path.join(out_path, f"{nama_asli}_pad.wav"), audio_pad, sr)
        else:
            jumlah_chunk = len(audio) // target_len
            for i in range(jumlah_chunk):
                start = i * target_len
                end = start + target_len
                chunk = audio[start:end]
                sf.write(os.path.join(out_path, f"{nama_asli}_part{i+1}.wav"), chunk, sr)

    jml_sesudah = len(os.listdir(out_path))
    
    print(f"{cls:<15} | {jml_sebelum:<10} | {jml_sesudah:<10}")

print("\nSelesai")

## Augmentasi

In [ ]:
base_dir = "dataset"
in_dir = os.path.join(base_dir, "durasi-2s")
out_dir = os.path.join(base_dir, "augmentasi")
classes = ["Clutch-Shoe", "Conecting-Rod", "Drive-Belt", "Piston", "Tensioner", "Slider", "Roller"]

os.makedirs(out_dir, exist_ok=True)
TARGET_JUMLAH = 40 

def aug_noise(audio):
    noise = np.random.randn(len(audio))
    return audio + 0.005 * noise, "noise"

def aug_pitch(audio, sr):
    n_steps = random.uniform(-2.0, 2.0)
    return librosa.effects.pitch_shift(y=audio, sr=sr, n_steps=n_steps), "pitch"

def aug_shift(audio):
    geser = int(random.uniform(-0.5, 0.5) * 16000)
    return np.roll(audio, geser), "shift"

print(f"Target augmentasi: Setiap kelas akan disamakan menjadi {TARGET_JUMLAH} file.\n")
print(f"{'KELAS':<15} | {'ASLI':<6} | {'TAMBAHAN':<10} | {'TOTAL':<6}")
print("-" * 50)

for cls in classes:
    in_path = os.path.join(in_dir, cls)
    out_path = os.path.join(out_dir, cls)
    os.makedirs(out_path, exist_ok=True)
    
    if not os.path.exists(in_path): continue
        
    files = sorted([f for f in os.listdir(in_path) if f.endswith('.wav')])
    jml_asli = len(files)
    
    if jml_asli == 0: continue
        
    indeks_file = 1
        
    for f in files:
        nama_baru_ori = f"{cls.lower()}-{indeks_file:03d}-ori.wav"
        shutil.copy(os.path.join(in_path, f), os.path.join(out_path, nama_baru_ori))
        indeks_file += 1
        
    selisih = TARGET_JUMLAH - jml_asli
    
    if selisih > 0:
        for i in range(selisih):
            file_sumber = files[i % jml_asli]
            audio, sr = librosa.load(os.path.join(in_path, file_sumber), sr=16000)
            
            pilihan = random.choice(['noise', 'pitch', 'shift'])
            
            if pilihan == 'noise':
                audio_aug, label_aug = aug_noise(audio)
            elif pilihan == 'pitch':
                audio_aug, label_aug = aug_pitch(audio, sr)
            else:
                audio_aug, label_aug = aug_shift(audio)
            
            nama_baru_aug = f"{cls.lower()}-{indeks_file:03d}-aug-{label_aug}.wav"
            sf.write(os.path.join(out_path, nama_baru_aug), audio_aug, sr)
            indeks_file += 1
            
    print(f"{cls:<15} | {jml_asli:<6} | {selisih:<10} | {TARGET_JUMLAH:<6}")

print("\nProses selesai! Data memiliki variasi metode augmentasi yang kaya dan panjang matriks terjamin aman.")

## Feature Extraction

# Feature Extraxtion

In [ ]:
in_dir = "dataset/augmentasi" 
classes = ["Clutch-Shoe", "Conecting-Rod", "Drive-Belt", "Piston", "Tensioner", "Slider", "Roller"]

X = []
y = []

print("Memulai Ekstraksi Fitur ke Mel-Spectrogram...\n")

for label_idx, cls in enumerate(classes):
    path = os.path.join(in_dir, cls)
    
    if not os.path.exists(path):
        continue
        
    files = sorted([f for f in os.listdir(path) if f.endswith('.wav')])
    print(f"Memproses kelas {cls:<15} ({len(files)} file)...")
    
    for f in files:
        file_path = os.path.join(path, f)
        audio, sr = librosa.load(file_path, sr=16000)
        mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128, hop_length=512)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        
        X.append(mel_spec_db)
        y.append(label_idx)

X = np.array(X)
y = np.array(y)
X = X[..., np.newaxis] 

print("\nProses Selesai!")
print("-" * 45)
print(f"Bentuk Matriks Data (X)  : {X.shape}")
print(f"Bentuk Matriks Label (y) : {y.shape}")
print("-" * 45)

np.save("X_data.npy", X)
np.save("y_label.npy", y)

print("File 'X_data.npy' dan 'y_label.npy' berhasil disimpan dan siap digunakan!")

## Model Training

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

print("=== MEMULAI FASE PELATIHAN MODEL CNN ===\n")

X = np.load("X_data.npy")
y = np.load("y_label.npy")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

y_train_encoded = to_categorical(y_train, num_classes=7)
y_test_encoded = to_categorical(y_test, num_classes=7)

print(f"Jumlah Data Latih : {X_train.shape[0]} sampel")
print(f"Jumlah Data Uji   : {X_test.shape[0]} sampel\n")


In [ ]:
dimensi_input = (X_train.shape[1], X_train.shape[2], X_train.shape[3])

model = Sequential([
    Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=dimensi_input),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(7, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("Memulai proses belajar (Epochs)...\n")

history = model.fit(
    X_train, 
    y_train_encoded, 
    epochs=1000, 
    batch_size=8, 
    validation_data=(X_test, y_test_encoded)
)

print("\nPelatihan Selesai!")

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Akurasi Latih', color='blue', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Akurasi Ujian', color='orange', linewidth=2)
plt.title('Grafik Akurasi Model')
plt.xlabel('Epochs (Putaran)')
plt.ylabel('Akurasi')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Kesalahan Latih', color='blue', linewidth=2)
plt.plot(history.history['val_loss'], label='Kesalahan Ujian', color='orange', linewidth=2)
plt.title('Grafik Tingkat Kesalahan (Loss)')
plt.xlabel('Epochs (Putaran)')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
model.save("Model_CNN_Suara_Mesin.h5")
print("\nModel telah berhasil disimpan dengan nama 'Model_CNN_Suara_Mesin.h5'")

# Inference 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model

print("=== MEMULAI EVALUASI DATA TESTING ===\n")

X = np.load("X_data.npy")
y = np.load("y_label.npy")
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = load_model("Model_CNN_Suara_Mesin.h5")

print("Model sedang memproses seluruh data ujian...\n")
prediksi_probabilitas = model.predict(X_test)
y_pred = np.argmax(prediksi_probabilitas, axis=1)

classes = ["Clutch-Shoe", "Conecting-Rod", "Drive-Belt", "Piston", "Tensioner", "Slider", "Roller"]
print("-" * 55)
print("RAPOR KINERJA MODEL PADA DATA TESTING (20% DATA)")
print("-" * 55)
print(classification_report(y_test, y_pred, target_names=classes))

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix: Hasil Tebakan vs Jawaban Asli', fontsize=14, pad=15)
plt.ylabel('Jawaban Asli (Data Sebenarnya)', fontsize=12)
plt.xlabel('Tebakan Model (Prediksi)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()